# Inspect audio SSMs

Interactive notebook to:

1. Optionally reuse a precomputed features directory.
2. Select one RWC-POP audio file.
3. Visualize the reference preview, available SSMs, and stored feature arrays.
4. Compute missing features on demand with the existing pipeline.


### Run and skip c:   (code for notebook, no interesting stuff here) 

In [ ]:
from __future__ import annotations

import json
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Image, Markdown, display

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "pipeline.py").exists():
    candidates = [
        PROJECT_DIR,
        PROJECT_DIR / "maestria/tesis/msa/MSA-with-Interpretable-Features",
    ]
    for candidate in candidates:
        if (candidate / "pipeline.py").exists():
            PROJECT_DIR = candidate.resolve()
            break
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from data_loading import FeatureConfig, FEATURE_NAME_ORDER, load_runtime_config
from pipeline import extract_all

CONFIG_PATH = PROJECT_DIR / "msa_feature_config.template.yaml"
raw_config, yaml_cfg = load_runtime_config(CONFIG_PATH)
DEFAULT_AUDIO_DIR = Path(raw_config.get("audio_dir", PROJECT_DIR / "data/rwc_p_20/audio")).expanduser()
DEFAULT_FEATURES_DIR = Path(raw_config.get("out_dir", PROJECT_DIR / "feature_outputs")).expanduser()
DEFAULT_TMP_DIR = PROJECT_DIR / "notebook_tmp"

REFERENCE_KEYS = {"sections", "fused"}
NON_ARRAY_KEYS = {
    "audio_path", "sr", "duration", "frame_times", "beat_boundaries", "beat_downbeat_boundaries",
    "stm_times", "stm_effective_window_s", "onset_env",
}


In [ ]:
def build_runtime_config(base_cfg: FeatureConfig, features_dir_text: str | None) -> tuple[FeatureConfig, Path | None]:
    features_dir = Path(features_dir_text).expanduser() if features_dir_text and features_dir_text.strip() else None
    cfg_payload = asdict(base_cfg)
    cfg_payload["sections_dir"] = Path(cfg_payload["sections_dir"]) if cfg_payload["sections_dir"] else None
    cfg_payload["ssl_models_root"] = Path(cfg_payload["ssl_models_root"])
    cfg_payload["matpac_checkpoint_path"] = Path(cfg_payload["matpac_checkpoint_path"]) if cfg_payload["matpac_checkpoint_path"] else None
    cfg_payload["musicfm_stat_path"] = Path(cfg_payload["musicfm_stat_path"]) if cfg_payload["musicfm_stat_path"] else None
    cfg_payload["musicfm_model_path"] = Path(cfg_payload["musicfm_model_path"]) if cfg_payload["musicfm_model_path"] else None
    return FeatureConfig(**cfg_payload), features_dir


def list_audio_files(audio_dir: Path) -> list[Path]:
    if not audio_dir.exists():
        return []
    return sorted(audio_dir.glob("*.wav"))


def parse_rwc_numeric_id(value: str | int | None) -> int | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    digits = "".join(ch for ch in text if ch.isdigit())
    return int(digits) if digits else None


def select_audio(audio_files: list[Path], rwc_audio_id: str | int | None = None, random_seed: int | None = None) -> Path:
    if not audio_files:
        raise FileNotFoundError("No WAV files found in the configured audio directory.")
    numeric_id = parse_rwc_numeric_id(rwc_audio_id)
    if numeric_id is not None:
        patterns = (
            f"{numeric_id:03d}",
            f"P{numeric_id:03d}",
            f"RM-P{numeric_id:03d}",
            f"RWC_P{numeric_id:03d}",
        )
        matches = [path for path in audio_files if any(pattern in path.stem for pattern in patterns)]
        if not matches:
            raise FileNotFoundError(f"No audio file matched RWC id {numeric_id:03d} in {audio_files[0].parent}")
        if len(matches) > 1:
            print("Multiple files matched the requested id; using the first one:")
            for path in matches:
                print(f"  - {path.name}")
        return matches[0]
    if random_seed is None:
        raise ValueError("Provide either `rwc_audio_id` or `random_seed`.")
    rng = np.random.default_rng(random_seed)
    return audio_files[int(rng.integers(0, len(audio_files)))]


def artifact_paths(audio_path: Path, features_dir: Path | None) -> dict[str, Path] | None:
    if features_dir is None:
        return None
    stem = audio_path.stem
    return {
        "npz": features_dir / f"{stem}_features.npz",
        "summary": features_dir / f"{stem}_summary.json",
        "preview_png": features_dir / f"{stem}_preview.png",
        "preview_svg": features_dir / f"{stem}_preview.svg",
    }


def has_precomputed(audio_path: Path, features_dir: Path | None) -> bool:
    paths = artifact_paths(audio_path, features_dir)
    return bool(paths and paths["npz"].exists())


def load_feature_bundle(npz_path: Path) -> dict[str, np.ndarray | str | float | int]:
    with np.load(npz_path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


def load_summary(summary_path: Path) -> dict:
    if not summary_path.exists():
        return {}
    return json.loads(summary_path.read_text(encoding="utf-8"))


def classify_keys(bundle: dict) -> tuple[list[str], list[str], list[str]]:
    ssm_keys = sorted(key for key in bundle if key.startswith("ssm_"))
    beat_keys = sorted(key for key in bundle if key.startswith("beat_"))
    raw_keys = sorted(
        key for key, value in bundle.items()
        if key not in NON_ARRAY_KEYS and not key.startswith("ssm_") and not key.startswith("beat_") and isinstance(value, np.ndarray)
    )
    return ssm_keys, beat_keys, raw_keys


def feature_name_from_ssm_key(key: str) -> str:
    return key.replace("ssm_", "", 1)


def selected_ssm_keys(bundle: dict, selected_features: tuple[str, ...]) -> list[str]:
    ssm_keys, _, _ = classify_keys(bundle)
    selected = []
    for key in ssm_keys:
        feature_name = feature_name_from_ssm_key(key)
        if feature_name in selected_features or feature_name in REFERENCE_KEYS:
            selected.append(key)
    return selected


def plot_selected_ssms(bundle: dict, selected_keys: list[str]) -> None:
    if not selected_keys:
        display(Markdown("No SSMs available for the selected features."))
        return
    n = len(selected_keys)
    cols = min(3, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows), squeeze=False)
    for ax, key in zip(axes.ravel(), selected_keys):
        matrix = np.asarray(bundle[key])
        ax.imshow(matrix, origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        ax.set_title(key)
        ax.set_xlabel("beat interval")
        ax.set_ylabel("beat interval")
    for ax in axes.ravel()[len(selected_keys):]:
        ax.axis("off")
    plt.show()


def show_feature_arrays(bundle: dict, selected_features: tuple[str, ...]) -> None:
    ssm_keys, beat_keys, raw_keys = classify_keys(bundle)
    rows = []
    for key in raw_keys + beat_keys + ssm_keys:
        base_name = key.replace("beat_", "").replace("ssm_", "")
        if base_name not in selected_features and base_name not in REFERENCE_KEYS:
            continue
        value = np.asarray(bundle[key])
        rows.append((key, value.shape, str(value.dtype)))
    if not rows:
        display(Markdown("No stored arrays match the selected features."))
        return
    html = ["<table>", "<tr><th>key</th><th>shape</th><th>dtype</th></tr>"]
    for key, shape, dtype in rows:
        html.append(f"<tr><td><code>{key}</code></td><td>{shape}</td><td>{dtype}</td></tr>")
    html.append("</table>")
    display(HTML("".join(html)))


def render_bundle(audio_path: Path, bundle: dict, summary: dict, selected_features: tuple[str, ...], paths: dict | None) -> None:
    display(Markdown(f"## {audio_path.name}"))
    if paths:
        preview_path = paths["preview_png"] if paths["preview_png"].exists() else paths["preview_svg"]
        if preview_path.exists() and preview_path.suffix.lower() == ".png":
            display(Markdown("### Reference preview"))
            display(Image(filename=str(preview_path)))
        elif preview_path.exists():
            display(Markdown(f"Preview available at `{preview_path}`"))
    if summary:
        display(Markdown("### Summary"))
        display(HTML(
            f"<pre>{json.dumps(summary, indent=2)}</pre>"
        ))
    display(Markdown("### SSMs"))
    plot_selected_ssms(bundle, selected_ssm_keys(bundle, selected_features))
    display(Markdown("### Stored feature arrays"))
    show_feature_arrays(bundle, selected_features)


## Audio inspection

### Song selection from RWC-POP dataset!!!

In [ ]:
# Set either `rwc_audio_id` or `random_seed`.
# Accepted id examples: 1, 001, P001, RM-P001, RWC_P001.
rwc_audio_id = 1
random_seed = 42

### Features inspection

In [ ]:
# Edit these values directly, then run this cell.

# 0. Optional: directory with precomputed *_features.npz, previews, and summaries.
features_dir_text = str(DEFAULT_FEATURES_DIR)
# Example: features_dir_text = ""

# 1. RWC-POP audio directory.
audio_dir_text = str(DEFAULT_AUDIO_DIR)

# 2. Feature/SSM selection.
selected_features = ("stm", "cens", "mfcc" ,"sections", "musicfm")

# 3. Missing-feature behavior.
# If False and the selected audio has no precomputed features, the cell stops with guidance.
compute_if_missing = False


In [ ]:
audio_dir = Path(audio_dir_text).expanduser()
audio_files = list_audio_files(audio_dir)
print(f"Found {len(audio_files)} audio files in {audio_dir}")

audio_path = select_audio(audio_files, rwc_audio_id=rwc_audio_id, random_seed=random_seed)
print(f"Selected audio: {audio_path.name}")

cfg, features_dir = build_runtime_config(yaml_cfg, features_dir_text)
paths = artifact_paths(audio_path, features_dir)
selected_features = tuple(selected_features)
if not selected_features:
    raise ValueError("`selected_features` must contain at least one feature name.")

if has_precomputed(audio_path, features_dir):
    print(f"Using precomputed features from {paths['npz']}")
else:
    out_dir = features_dir if features_dir is not None else DEFAULT_TMP_DIR
    if not compute_if_missing:
        raise FileNotFoundError(
            f"No precomputed features found for {audio_path.name}. "
            f"Set `compute_if_missing = True` to extract them into {out_dir}."
        )
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Computing missing features into {out_dir}...")
    extract_all(audio_path, out_dir, cfg)
    features_dir = out_dir
    paths = artifact_paths(audio_path, features_dir)
    print("Feature extraction finished.")

bundle = load_feature_bundle(paths["npz"])
summary = load_summary(paths["summary"])
render_bundle(audio_path, bundle, summary, selected_features, paths)


### Listening inspection

Select the beat interval or intervals to hear

In [ ]:
#

In [ ]:
#

## Ground-truth vs segmentation outputs

Set the experiment output folder below, then run the following cells after the feature/SSM inspection cell.


In [ ]:
# Experiment output directory produced by `run_structure_baseline.py`.
# It should contain one subfolder per algorithm, each with <track_id>.jams files.
experiment_outputs_dir = PROJECT_DIR / "experiments/structure_baseline/example_code_refactor2"

# SSM to use as the background for the comparison plots.
comparison_ssm_key = "ssm_fused" if "ssm_fused" in bundle else next(key for key in bundle if key.startswith("ssm_") and key != "ssm_sections")

# Set to None to show every algorithm folder with a JAMS for the selected song.
# Example: comparison_algorithms = ("as_seg_cbm", "msaf_foote")
comparison_algorithms = None


In [ ]:
from matplotlib import patches


def _scalar(value):
    if isinstance(value, np.ndarray) and value.shape == ():
        return value.item()
    return value


def _track_id_from_audio(audio_path: Path | str) -> str:
    return Path(str(audio_path)).stem


def _parse_chorus_file(path: Path) -> list[dict]:
    sections = []
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split("\t")
        if len(parts) != 3:
            continue
        start_cs, end_cs, label = parts
        start = int(start_cs) / 100.0
        end = int(end_cs) / 100.0
        sections.append({"time": start, "duration": max(0.0, end - start), "label": label.strip().strip('"')})
    return sections


def _candidate_sections_dirs(audio_path: Path, summary: dict) -> list[Path]:
    candidates = []
    summary_cfg = summary.get("config", {}) if isinstance(summary, dict) else {}
    raw_feature_cfg = raw_config.get("feature_config", {}) if isinstance(raw_config, dict) else {}
    for value in (summary_cfg.get("sections_dir"), raw_feature_cfg.get("sections_dir"), getattr(yaml_cfg, "sections_dir", None)):
        if value:
            candidates.append(Path(value).expanduser())
    candidates.extend([
        audio_path.parent / "CHORUS",
        audio_path.parent / "annotations" / "AIST.RWC-MDB-P-2001.CHORUS",
        PROJECT_DIR / "CHORUS",
        PROJECT_DIR / "annotations" / "AIST.RWC-MDB-P-2001.CHORUS",
    ])
    out = []
    for candidate in candidates:
        if candidate not in out:
            out.append(candidate)
    return out


def load_ground_truth_sections(audio_path: Path | str, summary: dict) -> tuple[list[dict], Path | None]:
    audio_path = Path(str(audio_path))
    track_id = _track_id_from_audio(audio_path)
    for sections_dir in _candidate_sections_dirs(audio_path, summary):
        candidate = sections_dir / f"{track_id}.CHORUS.TXT"
        if candidate.exists():
            return _parse_chorus_file(candidate), candidate
    return [], None


def _jams_annotation_data(jams_path: Path) -> list[dict]:
    payload = json.loads(jams_path.read_text(encoding="utf-8"))
    annotations = payload.get("annotations", [])
    if not annotations:
        return []
    return annotations[0].get("data", [])


def load_algorithm_segments(jams_path: Path) -> list[dict]:
    segments = []
    for idx, row in enumerate(_jams_annotation_data(jams_path), start=1):
        value = row.get("value", {}) if isinstance(row.get("value", {}), dict) else {}
        label = value.get("label", f"S{idx:02d}")
        start = float(row.get("time", 0.0))
        duration = float(row.get("duration", 0.0))
        segments.append({"time": start, "duration": max(0.0, duration), "label": label})
    return segments


def find_algorithm_jams(experiment_dir: Path, track_id: str, algorithms: tuple[str, ...] | None = None) -> dict[str, Path]:
    experiment_dir = Path(experiment_dir).expanduser()
    if not experiment_dir.exists():
        raise FileNotFoundError(f"Experiment output directory not found: {experiment_dir}")
    selected = set(algorithms) if algorithms else None
    found = {}
    for algorithm_dir in sorted(path for path in experiment_dir.iterdir() if path.is_dir()):
        if selected is not None and algorithm_dir.name not in selected:
            continue
        jams_path = algorithm_dir / f"{track_id}.jams"
        if jams_path.exists():
            found[algorithm_dir.name] = jams_path
    if not found:
        raise FileNotFoundError(f"No {track_id}.jams files found under {experiment_dir}")
    return found


def _boundary_times(segments: list[dict], duration: float) -> list[float]:
    boundaries = {0.0, float(duration)}
    for segment in segments:
        start = float(segment["time"])
        end = start + float(segment["duration"])
        boundaries.add(round(start, 6))
        boundaries.add(round(end, 6))
    return sorted(t for t in boundaries if 0.0 <= t <= duration + 1e-3)


def _label_colors(segment_sets: dict[str, list[dict]]):
    labels = []
    for segments in segment_sets.values():
        for segment in segments:
            label = str(segment.get("label", ""))
            if label not in labels:
                labels.append(label)
    cmap = plt.get_cmap("tab20", max(1, len(labels)))
    return {label: cmap(idx % cmap.N) for idx, label in enumerate(labels)}


def _draw_timeline(ax, segments: list[dict], colors: dict[str, tuple], duration: float, title: str):
    ax.set_xlim(0, duration)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_title(title, loc="left", fontsize=10)
    for segment in segments:
        start = float(segment["time"])
        end = min(duration, start + float(segment["duration"]))
        label = str(segment.get("label", ""))
        color = colors.get(label, "0.8")
        ax.axvspan(start, end, ymin=0.1, ymax=0.9, color=color, alpha=0.75, edgecolor="white", linewidth=0.6)
        if end - start >= max(3.0, duration * 0.025):
            ax.text((start + end) / 2, 0.5, label, ha="center", va="center", fontsize=8, clip_on=True)
    ax.grid(axis="x", alpha=0.2)


def _draw_ssm(ax, ssm: np.ndarray, duration: float, segments: list[dict], colors: dict[str, tuple], title: str):
    ax.imshow(ssm, origin="lower", aspect="auto", cmap="magma", extent=[0, duration, 0, duration])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("time (s)")
    for segment in segments:
        start = float(segment["time"])
        end = min(duration, start + float(segment["duration"]))
        label = str(segment.get("label", ""))
        color = colors.get(label, "white")
        rect = patches.Rectangle((start, start), end - start, end - start, facecolor=color, edgecolor=color, alpha=0.18, linewidth=1.0)
        ax.add_patch(rect)
    for boundary in _boundary_times(segments, duration):
        ax.axvline(boundary, color="white", alpha=0.35, linewidth=0.6)
        ax.axhline(boundary, color="white", alpha=0.35, linewidth=0.6)


def plot_ground_truth_vs_segmentations(bundle: dict, summary: dict, experiment_dir: Path, ssm_key: str, algorithms=None):
    audio_value = _scalar(bundle.get("audio_path", audio_path))
    selected_audio_path = Path(str(audio_value))
    track_id = _track_id_from_audio(selected_audio_path)
    duration = float(_scalar(bundle.get("duration", summary.get("duration_s", 0.0))))
    ssm = np.asarray(bundle[ssm_key])

    gt_sections, gt_path = load_ground_truth_sections(selected_audio_path, summary)
    jams_by_algorithm = find_algorithm_jams(Path(experiment_dir), track_id, tuple(algorithms) if algorithms else None)
    algorithm_segments = {name: load_algorithm_segments(path) for name, path in jams_by_algorithm.items()}

    segment_sets = {"Ground truth": gt_sections, **algorithm_segments}
    colors = _label_colors(segment_sets)

    if gt_path:
        display(Markdown(f"Ground truth: `{gt_path}`"))
    else:
        display(Markdown("Ground truth file not found. The plots will show only algorithm outputs."))
    display(Markdown("Algorithms: " + ", ".join(f"`{name}`" for name in algorithm_segments)))

    n_rows = max(1, len(segment_sets))
    fig, axes = plt.subplots(n_rows, 1, figsize=(16, 1.15 * n_rows + 1.2), sharex=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (name, segments) in zip(axes, segment_sets.items()):
        _draw_timeline(ax, segments, colors, duration, name)
    axes[-1].set_xlabel("time (s)")
    fig.suptitle(f"Timeline comparison: {track_id}", fontsize=13)
    plt.show()

    n_panels = len(segment_sets)
    n_cols = min(3, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.5 * n_cols, 5.0 * n_rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, segments) in zip(axes, segment_sets.items()):
        _draw_ssm(ax, ssm, duration, segments, colors, f"{ssm_key}: {name}")
    for ax in axes[n_panels:]:
        ax.axis("off")
    fig.suptitle(f"SSM comparison with filled section areas: {track_id}", fontsize=13)
    plt.show()

    return {"ground_truth_path": gt_path, "jams": jams_by_algorithm, "segments": segment_sets}


In [ ]:
comparison = plot_ground_truth_vs_segmentations(
    bundle=
    bundle,
    summary=summary,
    experiment_dir=experiment_outputs_dir,
    ssm_key=comparison_ssm_key,
    algorithms=comparison_algorithms,
)
